In [0]:
#read bronze tables
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"

orders_table = f"{catalog}.{schema_name}.bronze_orders"
products_table = f"{catalog}.{schema_name}.bronze_products"

orders = spark.table(orders_table)
products = spark.table(products_table)

print("Bronze orders:", orders.count())
print("Bronze products:", products.count())

In [0]:
#Identify invalid records
#Inspect missing IDs, duplicate IDs and invalid quantities.

orders.groupBy("order_id").count() \
    .filter(F.col("count") > 1) \
    .show()

orders.filter(F.col("customer_id").isNull()).show()

orders.filter(
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0)
).show()

In [0]:
#Deduplicate and validate
# Parse dates and remove repeated order IDs.
# Keep Bronze unchanged.

orders_clean = (
    orders
    .withColumn(
        "order_date",
        F.to_date("order_date", "yyyy-MM-dd")
    )
    .dropDuplicates(["order_id"])
)

# Separate valid and rejected rows.
valid_condition = (
    F.col("order_id").isNotNull() &
    F.col("order_date").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("quantity").isNotNull() &
    (F.col("quantity") > 0) &
    F.col("unit_price").isNotNull() &
    (F.col("unit_price") >= 0)
)

silver_orders = orders_clean.filter(valid_condition)

rejected_orders = orders_clean.filter(
    ~valid_condition
)

print("Silver valid orders:", silver_orders.count())
print("Rejected orders:", rejected_orders.count())

In [0]:
#Save Silver tables
silver_orders_table = (
    f"{catalog}.{schema_name}.silver_orders"
)

rejected_orders_table = (
    f"{catalog}.{schema_name}.rejected_orders"
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_orders_table)
)

(
    rejected_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(rejected_orders_table)
)

print("Silver tables saved.")

In [0]:
#Validate the result
saved_silver = spark.table(silver_orders_table)
saved_rejected = spark.table(rejected_orders_table)

assert saved_silver.count() == 9
assert saved_rejected.count() == 1

print("Silver validation passed.")
print("Valid orders:", saved_silver.count())
print("Rejected orders:", saved_rejected.count())

display(saved_silver.orderBy("order_id"))
display(saved_rejected)